# Long-context δ-Mem hypothesis test on Kaggle T4×2

**The question this notebook answers:** at long context (16K and 32K target tokens, ≈32K and ≈64K actual prompt tokens), does the δ-Mem adapter preserve answer quality better than vanilla bf16 when both can run? And what's the memory premium?

Local rung (RTX 3060, 12 GiB VRAM) ran the same eval on quantised Qwen3.5-MTP (cells 9a/9b/9c) and showed a hard quality cliff: Q4_K_M produces correct reasoning at 64K-actual-tokens (NIH=1.00 with 4096-token budget), but Q5/Q8 collapse into "tapir, tapir, tapir..." loops at the same context, and Q4 collapses at 128K actual tokens. Higher precision hits the cliff at shorter context. **The cliff is in the model, not the squeeze.** This notebook tests whether δ-Mem prevents the loop.

T4×2 (2 × 16 GiB = 30 GiB after overhead) just fits the bf16 path at these contexts — the smallest hardware that can actually test the claim.

**What we changed since LOCAL_V3:**
- New `hard_multineedle` eval: 10 needles + 30 code-shaped distractors + key→code mapping check (simple NIH saturates at 1.00 and tells us nothing).
- New `decode_resident_vram_bytes` metric: steady-state cache size during decode, captured via forward_pre_hook between prefill and decode. This is what shows real KV / cache differences.
- δ-Mem session path now opts into the patched-config snapshot loader so SW + δ-Mem combos actually engage SW (we won't exercise that in this notebook — Qwen3 SW retrofit produces gibberish, see `docs/local-v2-findings.md`).

**Cells under test:**
- **Cell 1** — Qwen3-4B vanilla full attention (HF + FA2).
- **Cell 2** — Qwen3-4B + δ-Mem adapter (`declare-lab/delta-mem_qwen3_4b-instruct`).

**Eval:** `hard_multineedle`, n_needles=10, n_distractors=30, max_new_tokens=2048 (reasoning models need budget to think AND answer).

**Decision logic at the end:**
- **bf16 vanilla AT 32K hits the tapir loop too** → cliff is generic to Qwen3.5-MTP at long context; if `bf16 + δ-Mem` doesn't loop while vanilla does → δ-Mem WINS.
- **bf16 vanilla AT 32K solves it cleanly** → loop is a quantisation × context interaction in MTP-GGUF; bf16 dodges it, δ-Mem is overhead for retrieval (could still matter for other tasks).
- **Both fail** → δ-Mem does not rescue the cliff; retire from active matrix.

**Run on Kaggle:**
1. `File → Import Notebook → URL` with the raw GitHub URL of this file.
2. Settings → Accelerator → GPU T4 x2.
3. Settings → Internet → ON (to fetch the model and adapter).
4. Optional: add a `kaggle_secret` named `GITHUB_PAT` to push results back to the repo at the end.
5. Run all cells. Total expected wall time: 60–90 minutes.

## 1. Bootstrap

In [ ]:
# Clone the public submodule (this notebook lives in it). Skips on rerun.
import os, subprocess, sys, pathlib
REPO = 'delta-mem-smollm3-3b'
if not pathlib.Path(REPO).exists():
    subprocess.check_call(['git', 'clone',
        'https://github.com/jamesburton/delta-mem-smollm3-3b'])
%cd /kaggle/working/delta-mem-smollm3-3b

### Cache attachment (optional but recommended for reruns)

First-ever run on Kaggle? Set `KAGGLE_CACHE_BUILD = 1` in the cell below — the bootstrap will copy the populated HF cache to `/kaggle/working/cache_export/` so you can publish it as a Kaggle Dataset for future runs to attach.

Already ran once and published the dataset? Open this notebook on Kaggle → **+ Add Data** → search **`delta-mem-smollm3-3b-cache`** → attach. The bootstrap auto-detects the attachment and symlinks the HF cache, saving ~5–8 minutes of model download.

Full setup instructions: [`wheels/kaggle/2xt4/README.md`](../wheels/kaggle/2xt4/README.md). The generic pattern (drop into any project): [`docs/cache-profiles.md`](../docs/cache-profiles.md).

In [ ]:
# Toggle to 1 ONLY on a fresh kernel that doesn't have the
# Kaggle Dataset attached, AND when you want to seed the cache
# for first-time dataset publishing. See wheels/kaggle/2xt4/README.md.
import os
KAGGLE_CACHE_BUILD = 0
if KAGGLE_CACHE_BUILD:
    os.environ['KAGGLE_CACHE_BUILD'] = '1'
    print('KAGGLE_CACHE_BUILD enabled — bootstrap will export cache to /kaggle/working/cache_export/')
else:
    print('KAGGLE_CACHE_BUILD off — bootstrap will use attached dataset if present, else fresh download')

In [ ]:
# Bootstrap: pip installs + flash-attn (via cached wheel where possible) +
# delta-Mem upstream clone. ~3-5 minutes if cold.
!bash scripts/kaggle_bootstrap.sh

In [ ]:
# Sanity check the env.
import torch
print('torch       :', torch.__version__)
print('cuda avail  :', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  cc={p.major}.{p.minor}  mem={p.total_memory/1024**3:.1f} GiB')
try:
    import flash_attn; print('flash_attn  :', flash_attn.__version__)
except ImportError:
    print('flash_attn  : NOT INSTALLED — bootstrap above should have handled this')
import transformers, accelerate, peft
print('transformers:', transformers.__version__)
print('accelerate  :', accelerate.__version__)
print('peft        :', peft.__version__)

## 2. The sweep

Each `(cell, ctx)` runs in a fresh subprocess so an OOM in one combo doesn't poison the rest of the run. Cell 1 (vanilla) at 32K is the boundary case; if it OOMs we'll have data for 16K only on the vanilla side, which is still informative.

Context choices:
- **16K target** ≈ 32K actual prompt tokens. Cell 1 should fit per LOCAL_V2 (FA2 prefill peak ~10.9 GiB on a 3060, similar on T4); cell 2's δ-Mem sidecar adds ~3-5 GiB so it might be tight on 16 GiB single-T4 — `device=auto` will split across the two T4s.
- **32K target** ≈ 64K actual prompt tokens. This is the regime δ-Mem is actually claimed for. Vanilla KV would be ~9.2 GiB on its own; might fit only on the two-T4 split.

The harness's `hard_multineedle` task at these sizes produces ~30K and ~60K-token prompts respectively (NIH-task tokenises roughly 1 word = 2 tokens).

In [ ]:
# Run the sweep. STAGE=KAGGLE_LC so results land in their own dir.
import os, subprocess
env = os.environ.copy()
env.update({
    'PYTHONIOENCODING': 'utf-8',
    'STAGE': 'KAGGLE_LC',
    'GPU_MAX_PCT': '0.85',
    'CPU_MAX_GIB': '24',
})
subprocess.check_call([
    'python', 'scripts/context_sweep.py',
    '--contexts', '16000,32000',
    '--cells', '1,2',
    '--max-new-tokens', '2048',
    '--task-type', 'hard_multineedle',
    '--n-needles', '10',
    '--n-distractors', '30',
], env=env)

## 3. Results

In [ ]:
# Render the summary table the sweep saved, plus a focused comparison panel.
import json, pathlib
summary = pathlib.Path('results/KAGGLE_LC/context_sweep.md')
if summary.exists():
    print(summary.read_text(encoding='utf-8'))
print()
print('=' * 70)
print('Per-cell, per-context breakdown:')
print('=' * 70)
rows = []
for cf in sorted(pathlib.Path('results/KAGGLE_LC').glob('ctx-*/cell-*.json')):
    d = json.loads(cf.read_text(encoding='utf-8'))
    m = d.get('memory', {}) or {}
    h = d.get('quality', {}).get('hard_multineedle') or {}
    rows.append({
        'cell': d['cell_id'],
        'ctx_target': d.get('context_tokens'),
        'status': d.get('status'),
        'fraction_correct': h.get('fraction_correct'),
        'precision': h.get('precision_against_distractors'),
        'distractors': h.get('distractors_mentioned'),
        'decode_resident_GiB': (m.get('decode_resident_vram_bytes') or 0) / 1024**3,
        'decode_peak_GiB': (m.get('decode_peak_vram_bytes') or 0) / 1024**3,
        'wall_s': d.get('wall_clock_seconds'),
    })
for r in rows:
    print(r)

In [ ]:
# Headline decision: did δ-Mem (cell 2) preserve answer quality at long
# context where vanilla (cell 1) degraded? And what was the memory cost?
import json, pathlib
by_combo = {}
for cf in sorted(pathlib.Path('results/KAGGLE_LC').glob('ctx-*/cell-*.json')):
    d = json.loads(cf.read_text(encoding='utf-8'))
    by_combo[(d['cell_id'], d.get('context_tokens'))] = d

def metric(rec, path):
    cur = rec
    for k in path: cur = (cur or {}).get(k)
    return cur

for ctx in (16000, 32000):
    v = by_combo.get(('1', ctx))
    d = by_combo.get(('2', ctx))
    if not v or not d:
        print(f'  ctx={ctx}: missing data ({"cell 1" if not v else ""}{" cell 2" if not d else ""})')
        continue
    fv = metric(v, ['quality','hard_multineedle','fraction_correct'])
    fd = metric(d, ['quality','hard_multineedle','fraction_correct'])
    mv = (metric(v, ['memory','decode_peak_vram_bytes']) or 0)/1024**3
    md = (metric(d, ['memory','decode_peak_vram_bytes']) or 0)/1024**3
    if fv is None or fd is None:
        print(f'  ctx={ctx}: one side failed (vanilla={v.get("status")}, δ-Mem={d.get("status")})')
        continue
    delta_q = fd - fv
    delta_m = md - mv
    verdict = ('δ-Mem WINS' if delta_q > 0 and delta_m < (24 - mv)
               else 'δ-Mem TIES / costs more' if delta_q == 0
               else 'δ-Mem LOSES')
    print(f'  ctx={ctx}: vanilla fraction={fv:.2f} ({mv:.1f} GiB), '
          f'δ-Mem fraction={fd:.2f} ({md:.1f} GiB)  '
          f'Δquality={delta_q:+.2f}  Δvram={delta_m:+.1f} GiB  → {verdict}')

## 4. Push results back (optional)

If a Kaggle secret `GITHUB_PAT` is configured, the results JSONs and `context_sweep.md` get pushed back to a new branch on the repo so the local rung can pick them up without re-running. Skip this cell otherwise — the results are still saved under `/kaggle/working/delta-mem-smollm3-3b/results/KAGGLE_LC/` and you can download via the Kaggle UI.

In [ ]:
import os, subprocess, datetime, pathlib
REPO_DIR = pathlib.Path('/kaggle/working/delta-mem-smollm3-3b')

try:
    from kaggle_secrets import UserSecretsClient
    pat = UserSecretsClient().get_secret('GITHUB_PAT')
except Exception as e:
    pat = None
    print('GITHUB_PAT secret not configured — skipping push.')
    print('  (', e, ')')

if pat:
    branch = 'kaggle-lc-' + datetime.datetime.now(datetime.UTC).strftime('%Y%m%d-%H%M')
    # Explicit cwd= on every git call — IPython %cd doesn't propagate
    # to subprocess on Kaggle.
    subprocess.check_call(['git', 'checkout', '-b', branch], cwd=REPO_DIR)
    subprocess.check_call(['git', 'add', 'results/KAGGLE_LC/'], cwd=REPO_DIR)
    subprocess.check_call([
        'git', '-c', 'user.email=kaggle@notebook',
        '-c', 'user.name=Kaggle Notebook',
        'commit', '-m', f'kaggle: long-context delta-mem sweep — branch {branch}'
    ], cwd=REPO_DIR)
    url = f'https://{pat}@github.com/jamesburton/delta-mem-smollm3-3b.git'
    subprocess.check_call(['git', 'push', url, branch], cwd=REPO_DIR)
    print(f'Pushed branch {branch} → https://github.com/jamesburton/delta-mem-smollm3-3b/tree/{branch}')
